In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/march-machine-learning-mania-2026/Conferences.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WNCAATourneyDetailedResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WRegularSeasonCompactResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MNCAATourneySeedRoundSlots.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MRegularSeasonDetailedResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MNCAATourneyCompactResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MGameCities.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WSecondaryTourneyCompactResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WGameCities.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MSeasons.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WNCAATourneySlots.csv
/kaggle/input/competitions/march-machine-learning

In [2]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional
import warnings
import os
import gc
from tqdm import tqdm
warnings.filterwarnings('ignore')

# ==================== 1. 环境与GPU检查 ====================
try:
    from numba import cuda, njit
    HAS_GPU = cuda.is_available()
    GPU_COUNT = len(cuda.gpus) if HAS_GPU else 0
    print(f"✅ 检测到GPU环境：可用GPU数量 {GPU_COUNT}")
    if GPU_COUNT > 0:
        for i in range(GPU_COUNT):
            print(f"  - GPU{i}: {cuda.gpus[i].name}")
except Exception as e:
    HAS_GPU = False
    GPU_COUNT = 0
    print(f"⚠️  GPU不可用，自动切换到优化CPU版本：{str(e)}")

# ==================== 2. 固定配置（Kaggle比赛专用） ====================
BASE_DATA_PATH = '/kaggle/input/competitions/march-machine-learning-mania-2026/'
OUTPUT_PATH = '/kaggle/working/doubao_features.parquet'
MIN_SEASON = 2003
MAX_SEASON = 2025

# ==================== 3. 数据文件检查 ====================
required_files = [
    'MRegularSeasonDetailedResults.csv',
    'WRegularSeasonDetailedResults.csv',
    'MNCAATourneyDetailedResults.csv',   # 新增：男子锦标赛详细数据
    'WNCAATourneyDetailedResults.csv',   # 新增：女子锦标赛详细数据
    'MNCAATourneySeeds.csv',
    'WNCAATourneySeeds.csv'
]

print("\n🔍 检查数据文件是否存在...")
all_files_exist = True
for file in required_files:
    full_path = os.path.join(BASE_DATA_PATH, file)
    if os.path.exists(full_path):
        print(f"✅ 找到文件: {file}")
    else:
        print(f"❌ 缺失文件: {full_path}")
        all_files_exist = False

if not all_files_exist:
    raise FileNotFoundError("数据文件路径错误，请检查BASE_DATA_PATH配置")

# ==================== 4. GPU加速函数 ====================
@njit
def incremental_elo_calculation(
    day_nums: np.ndarray,
    scores: np.ndarray,
    opp_scores: np.ndarray,
    is_home: np.ndarray,
    is_playoff: np.ndarray,
    init_elo: float = 1500.0,
    home_advantage: float = 65.0,
    regular_k: float = 20.0,
    playoff_k: float = 30.0
) -> np.ndarray:
    n_games = len(day_nums)
    pre_game_elo = np.zeros(n_games, dtype=np.float32)
    current_elo = init_elo
    
    for i in range(n_games):
        pre_game_elo[i] = current_elo
        if i < n_games - 1:
            k = playoff_k if is_playoff[i] else regular_k
            elo_a = current_elo + (home_advantage if is_home[i] == 1 else 0)
            elo_b = 1500.0
            actual_a = 1 if scores[i] > opp_scores[i] else 0
            expected_a = 1 / (1 + 10 ** ((elo_b - elo_a) / 400))
            current_elo += k * (actual_a - expected_a)
            if day_nums[i+1] < day_nums[i]:
                current_elo = 0.75 * current_elo + 0.25 * init_elo
    
    return pre_game_elo

@njit
def incremental_sos_calculation(
    day_nums: np.ndarray,
    opp_team_ids: np.ndarray,
    team_win_flags: np.ndarray
) -> np.ndarray:
    n_games = len(day_nums)
    sos_values = np.zeros(n_games, dtype=np.float32)
    
    for i in range(n_games):
        if i == 0:
            sos_values[i] = 0.5
        else:
            opp_win_rates = np.zeros(i, dtype=np.float32)
            for j in range(i):
                opp_win_rates[j] = np.mean(team_win_flags[0:j+1]) if j > 0 else 0.5
            avg_opp_win_rate = np.mean(opp_win_rates) if i > 0 else 0.5
            sos_values[i] = 0.75 * avg_opp_win_rate + 0.25 * 0.5
    
    return sos_values

# ==================== 5. 数据泄露检测（修复误判规则，添加非时序特征） ====================
class DataLeakageGuard:
    def __init__(self, data: pd.DataFrame, date_col: str = 'DayNum', team_col: str = 'TeamID', season_col: str = 'Season'):
        self.data = data.sort_values([season_col, team_col, date_col]).reset_index(drop=True)
        self.date_col = date_col
        self.team_col = team_col
        self.season_col = season_col
        # 所有特征的合理初始值（第一场比赛的默认值）
        self.valid_initial_values = {
            'pre_game_elo': 1500.0, 'opp_pre_game_elo': 1500.0, 'sos': 0.5, 'adj_em': 0.0,
            'team_seed': 0.0, 'opp_seed': 0.0, 'seed_diff': 0.0,
            'rolling_5game_mean_Score': 0.0, 'rolling_10game_mean_Score': 0.0,
            'rolling_5game_mean_OppScore': 0.0, 'rolling_10game_mean_OppScore': 0.0,
            'rolling_5game_mean_FGM': 0.0, 'rolling_10game_mean_FGM': 0.0,
            'rolling_5game_mean_FGA': 0.0, 'rolling_10game_mean_FGA': 0.0,
            'rolling_5game_mean_FGM3': 0.0, 'rolling_10game_mean_FGM3': 0.0,
            'rolling_5game_mean_FGA3': 0.0, 'rolling_10game_mean_FGA3': 0.0,
            'rolling_5game_mean_FTM': 0.0, 'rolling_10game_mean_FTM': 0.0,
            'rolling_5game_mean_FTA': 0.0, 'rolling_10game_mean_FTA': 0.0,
            'rolling_5game_mean_TO': 0.0, 'rolling_10game_mean_TO': 0.0,
            'rolling_5game_mean_OR': 0.0, 'rolling_10game_mean_OR': 0.0,
            'rolling_5game_mean_DR': 0.0, 'rolling_10game_mean_DR': 0.0,
            'rolling_10game_win_rate': 0.0,
            'rolling_10game_mean_eFG_pct': 0.0, 'rolling_10game_mean_TOV_pct': 0.0,
            'rolling_10game_mean_ORB_pct': 0.0, 'rolling_10game_mean_FT_rate': 0.0,
            'elo_diff': 0.0,
            # v8新增：对手节奏统计（tempo_diff所需），合理初始值=0.0（shift+fillna(0)）
            'rolling_5game_mean_OppFGA': 0.0,  'rolling_10game_mean_OppFGA': 0.0,
            'rolling_5game_mean_OppOR': 0.0,   'rolling_10game_mean_OppOR': 0.0,
            'rolling_5game_mean_OppTO': 0.0,   'rolling_10game_mean_OppTO': 0.0,
            'rolling_5game_mean_OppFTA': 0.0,  'rolling_10game_mean_OppFTA': 0.0,
        }
        # 非时序特征：不需要检查滚动泄露的特征（包括对手Elo和Elo差值，因为它们依赖于对手的历史数据）
        self.non_temporal_features = ['is_mens', 'opp_pre_game_elo', 'elo_diff']

    def check_rolling_leakage(self, feature_col: str) -> bool:
        if feature_col in self.non_temporal_features:
            return True
        
        grouped = self.data.groupby([self.season_col, self.team_col])
        for _, group in grouped:
            if len(group) < 1:
                continue
            
            feature_vals = group[feature_col].values
            non_null_mask = ~np.isnan(feature_vals)
            if non_null_mask.any():
                first_non_null_idx = np.argmax(non_null_mask)
                first_non_null_val = feature_vals[first_non_null_idx]
                
                if first_non_null_idx == 0:
                    if feature_col in self.valid_initial_values:
                        if abs(first_non_null_val - self.valid_initial_values[feature_col]) < 1e-6:
                            continue
                        else:
                            print(f"⚠️ 数据泄露：{feature_col} 第一场值{first_non_null_val}≠合理初始值{self.valid_initial_values[feature_col]}")
                            return False
                    else:
                        print(f"⚠️ 数据泄露：{feature_col} 在{group.iloc[0][[self.season_col, self.team_col, self.date_col]]}处提前有值（无合理初始值）")
                        return False
                else:
                    if np.all(non_null_mask[:first_non_null_idx]):
                        print(f"⚠️ 数据泄露：{feature_col} 在{group.iloc[first_non_null_idx][[self.season_col, self.team_col, self.date_col]]}前无NaN，疑似用未来数据")
                        return False
        return True

    def validate_all_features(self, feature_cols: List[str]) -> bool:
        all_valid = True
        temporal_features = [col for col in feature_cols if col not in self.non_temporal_features]
        for col in temporal_features:
            if not self.check_rolling_leakage(col):
                all_valid = False
        if all_valid:
            print("✅ 所有时序特征无数据泄露（合理初始值已验证）")
        else:
            print("❌ 检测到数据泄露，请检查特征计算逻辑")
        return all_valid

# ==================== 6. 滚动特征工程类（修复分组内shift） ====================
class RollingFeatureEngineer:
    def __init__(self, window_sizes: List[int] = [5, 10], stats: List[str] = ['mean']):
        self.window_sizes = window_sizes
        self.stats = stats
    
    def _prepare_team_games(self, game_data: pd.DataFrame) -> pd.DataFrame:
        team_cols = ['Season', 'TeamID', 'OppTeamID', 'DayNum', 'Score', 'OppScore',
                     'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'TO', 'OR', 'DR',
                     'OppFGM', 'OppFGA', 'OppFGM3', 'OppFGA3', 'OppFTM', 'OppFTA', 'OppTO', 'OppOR', 'OppDR',
                     'IsHome', 'IsPlayoff', 'is_mens']
        missing_cols = [col for col in team_cols if col not in game_data.columns]
        if missing_cols:
            raise ValueError(f"缺少必要列：{missing_cols}")
        
        team_games = game_data[team_cols].sort_values(['Season', 'TeamID', 'DayNum']).reset_index(drop=True)
        return team_games
    
    def calculate_rolling_stats(self, game_data: pd.DataFrame) -> pd.DataFrame:
        team_games = self._prepare_team_games(game_data)
        feature_cols = []
        
        base_metrics = {
            'Score': '得分', 'OppScore': '对手得分', 'FGM': '命中数', 'FGA': '出手数',
            'FGM3': '三分命中数', 'FGA3': '三分出手数', 'FTM': '罚球命中数', 'FTA': '罚球出手数',
            'TO': '失误数', 'OR': '进攻篮板', 'DR': '防守篮板',
            # v8新增：对手节奏统计，用于主脚本计算 opp_pace_estimate → tempo_diff
            'OppFGA': '对手出手数', 'OppOR': '对手进攻篮板',
            'OppTO': '对手失误数', 'OppFTA': '对手罚球出手数',
        }
        
        grouped = team_games.groupby(['Season', 'TeamID'])
        
        for metric in tqdm(base_metrics.keys(), desc="📊 计算滚动统计特征"):
            for window in self.window_sizes:
                for stat in self.stats:
                    col_name = f'rolling_{window}game_{stat}_{metric}'
                    if stat == 'mean':
                        # 分组内滚动计算均值
                        rolling_val = grouped[metric].rolling(window=window, min_periods=1).mean()
                        # 分组内shift(1)，确保第一场比赛的滚动值是NaN（无历史数据）
                        rolling_val = rolling_val.groupby(level=[0,1]).shift(1)
                        # 填充初始值为0.0
                        rolling_val = rolling_val.reset_index(drop=True).fillna(0.0).astype(np.float32)
                        team_games[col_name] = rolling_val
                    feature_cols.append(col_name)
        
        # 计算10场胜率（分组内shift(1)）
        team_games['is_win'] = (team_games['Score'] > team_games['OppScore']).astype(np.int32)
        win_rate = grouped['is_win'].rolling(window=10, min_periods=1).mean()
        # 分组内shift(1)
        win_rate = win_rate.groupby(level=[0,1]).shift(1)
        win_rate = win_rate.reset_index(drop=True).fillna(0.0).astype(np.float32)
        team_games['rolling_10game_win_rate'] = win_rate
        feature_cols.append('rolling_10game_win_rate')
        
        return team_games, feature_cols

# ==================== 7. 四大要素计算（修复分组内shift） ====================
def extract_seed(seed_str: str) -> int:
    if pd.isna(seed_str):
        return 0
    seed_num = ''.join([c for c in seed_str if c.isdigit()])
    return int(seed_num) if seed_num else 0

def calculate_four_factors(game_data: pd.DataFrame) -> pd.DataFrame:
    game_data = game_data.copy().reset_index(drop=True)
    
    game_data['eFG_pct'] = ((game_data['FGM'] + 0.5 * game_data['FGM3']) / game_data['FGA']).astype(np.float32)
    game_data['TOV_pct'] = (game_data['TO'] / (game_data['FGA'] + 0.44 * game_data['FTA'] + game_data['TO'])).astype(np.float32)
    game_data['ORB_pct'] = (game_data['OR'] / (game_data['OR'] + game_data['OppDR'])).astype(np.float32)
    game_data['FT_rate'] = (game_data['FTA'] / game_data['FGA']).astype(np.float32)
    
    grouped = game_data.groupby(['Season', 'TeamID'])
    four_factor_cols = []
    for factor in tqdm(['eFG_pct', 'TOV_pct', 'ORB_pct', 'FT_rate'], desc="🏀 计算四大要素特征"):
        col_name = f'rolling_10game_mean_{factor}'
        # 分组内滚动计算均值
        rolling_val = grouped[factor].rolling(window=10, min_periods=1).mean()
        # 分组内shift(1)
        rolling_val = rolling_val.groupby(level=[0,1]).shift(1)
        rolling_val = rolling_val.reset_index(drop=True).fillna(0.0).astype(np.float32)
        game_data[col_name] = rolling_val
        four_factor_cols.append(col_name)
    
    for col in four_factor_cols:
        game_data[col] = game_data[col].replace([np.inf, -np.inf], 0.0)
    
    return game_data, four_factor_cols

# ==================== 8. 数据加载与预处理 ====================
def load_and_preprocess_data(base_path: str):
    print("\n📥 加载数据...")

    # ── 常规赛 ──────────────────────────────────────────────────────────
    m_regular = pd.read_csv(os.path.join(base_path, 'MRegularSeasonDetailedResults.csv'))
    w_regular = pd.read_csv(os.path.join(base_path, 'WRegularSeasonDetailedResults.csv'))
    m_regular = m_regular[(m_regular['Season'] >= MIN_SEASON) & (m_regular['Season'] <= MAX_SEASON)]
    w_regular = w_regular[(w_regular['Season'] >= MIN_SEASON) & (w_regular['Season'] <= MAX_SEASON)]
    m_regular['IsPlayoff'] = False
    w_regular['IsPlayoff'] = False

    # ── 锦标赛详细数据（关键修复：训练数据必须包含锦标赛！） ────────────
    m_tourney = pd.read_csv(os.path.join(base_path, 'MNCAATourneyDetailedResults.csv'))
    w_tourney = pd.read_csv(os.path.join(base_path, 'WNCAATourneyDetailedResults.csv'))
    m_tourney = m_tourney[(m_tourney['Season'] >= MIN_SEASON) & (m_tourney['Season'] <= MAX_SEASON)]
    w_tourney = w_tourney[(w_tourney['Season'] >= MIN_SEASON) & (w_tourney['Season'] <= MAX_SEASON)]
    m_tourney['IsPlayoff'] = True
    w_tourney['IsPlayoff'] = True

    print(f"  常规赛行数: M={len(m_regular)}  W={len(w_regular)}")
    print(f"  锦标赛行数: M={len(m_tourney)}  W={len(w_tourney)}")

    # ── 合并常规赛 + 锦标赛 ─────────────────────────────────────────────
    m_all = pd.concat([m_regular, m_tourney], ignore_index=True)
    w_all = pd.concat([w_regular, w_tourney], ignore_index=True)
    m_all['is_mens'] = 1
    w_all['is_mens'] = 0
    regular = pd.concat([m_all, w_all], ignore_index=True)

    # ── 种子数据 ────────────────────────────────────────────────────────
    m_seeds = pd.read_csv(os.path.join(base_path, 'MNCAATourneySeeds.csv'))
    w_seeds = pd.read_csv(os.path.join(base_path, 'WNCAATourneySeeds.csv'))
    m_seeds['is_mens'] = 1
    w_seeds['is_mens'] = 0
    seeds = pd.concat([m_seeds, w_seeds], ignore_index=True)

    print("🔄 转换为球队视角数据（IsPlayoff 列随行传递）...")
    # winner 视角：IsPlayoff 已在 regular 里，rename 不触碰它
    winner_view = regular.rename(columns={
        'WTeamID': 'TeamID', 'LTeamID': 'OppTeamID',
        'WScore': 'Score', 'LScore': 'OppScore',
        'WFGM': 'FGM', 'WFGA': 'FGA', 'WFGM3': 'FGM3', 'WFGA3': 'FGA3',
        'WFTM': 'FTM', 'WFTA': 'FTA', 'WOR': 'OR', 'WDR': 'DR',
        'WTO': 'TO', 'WStl': 'Stl', 'WBlk': 'Blk', 'WPF': 'PF',
        'LFGM': 'OppFGM', 'LFGA': 'OppFGA', 'LFGM3': 'OppFGM3', 'LFGA3': 'OppFGA3',
        'LFTM': 'OppFTM', 'LFTA': 'OppFTA', 'LOR': 'OppOR', 'LDR': 'OppDR',
        'LTO': 'OppTO', 'LStl': 'OppStl', 'LBlk': 'OppBlk', 'LPF': 'OppPF'
    })
    winner_view['IsHome'] = (winner_view['WLoc'] == 'H').astype(np.int8)

    # loser 视角：IsPlayoff 同样随行传递
    loser_view = regular.rename(columns={
        'LTeamID': 'TeamID', 'WTeamID': 'OppTeamID',
        'LScore': 'Score', 'WScore': 'OppScore',
        'LFGM': 'FGM', 'LFGA': 'FGA', 'LFGM3': 'FGM3', 'LFGA3': 'FGA3',
        'LFTM': 'FTM', 'LFTA': 'FTA', 'LOR': 'OR', 'LDR': 'DR',
        'LTO': 'TO', 'LStl': 'Stl', 'LBlk': 'Blk', 'LPF': 'PF',
        'WFGM': 'OppFGM', 'WFGA': 'OppFGA', 'WFGM3': 'OppFGM3', 'WFGA3': 'OppFGA3',
        'WFTM': 'OppFTM', 'WFTA': 'OppFTA', 'WOR': 'OppOR', 'WDR': 'OppDR',
        'WTO': 'OppTO', 'WStl': 'OppStl', 'WBlk': 'OppBlk', 'WPF': 'OppPF'
    })
    loser_view['IsHome'] = (loser_view['WLoc'] == 'A').astype(np.int8)

    game_data = pd.concat([winner_view, loser_view], ignore_index=True)
    # ⚠️ 注意：IsPlayoff 已从 regular 正确继承，严禁在此处覆写为 False！
    # 将 bool 显式转为 int8，防止后续 int_cols 批量转换时因 dtype 识别问题丢失
    game_data['IsPlayoff'] = game_data['IsPlayoff'].astype(np.int8)
    
    print("💾 优化数据类型...")
    int_cols = game_data.select_dtypes(include=['int64']).columns
    float_cols = game_data.select_dtypes(include=['float64']).columns
    game_data[int_cols] = game_data[int_cols].astype('int32')
    game_data[float_cols] = game_data[float_cols].astype('float32')
    
    # 检查并处理可能的NaN值
    print("🔍 检查并处理NaN值...")
    na_cols = game_data[['Season', 'TeamID', 'OppTeamID', 'DayNum']].isna().sum()
    if na_cols.any():
        print(f"⚠️ 发现NaN值：{na_cols}，正在删除...")
        game_data = game_data.dropna(subset=['Season', 'TeamID', 'OppTeamID', 'DayNum'])
    
    del m_regular, w_regular, m_tourney, w_tourney, m_all, w_all, winner_view, loser_view
    gc.collect()
    
    print(f"✅ 数据加载完成，总比赛行数：{len(game_data)}")
    return game_data, seeds

# ==================== 9. 主特征构建函数（修复所有泄露问题+改用等值连接匹配Elo） ====================
def build_features_gpu(game_data: pd.DataFrame, seeds_data: pd.DataFrame):
    print("\n⚙️ 开始构建全量特征（准确率优先模式）...")
    
    # 1. 滚动基础特征（修复分组内shift，避免跨球队污染）
    rolling_engineer = RollingFeatureEngineer(window_sizes=[5, 10], stats=['mean'])
    team_games, rolling_cols = rolling_engineer.calculate_rolling_stats(game_data)
    gc.collect()
    
    # 2. 四大要素特征（修复分组内shift）
    team_games, four_factor_cols = calculate_four_factors(team_games)
    gc.collect()
    
    # 3. GPU加速Elo计算（修复对手Elo的未来数据问题+改用等值连接）
    print("\n🧮 计算Elo评级（GPU加速）...")
    team_games = team_games.sort_values(['Season', 'TeamID', 'DayNum']).reset_index(drop=True)
    grouped_elo = team_games.groupby(['Season', 'TeamID'])
    
    # 计算每个球队的pre_game_elo
    team_elo_list = []
    for (season, team_id), group in tqdm(grouped_elo, desc="Elo计算进度", total=len(grouped_elo)):
        group = group.sort_values('DayNum')
        elo_vals = incremental_elo_calculation(
            day_nums=group['DayNum'].values,
            scores=group['Score'].values,
            opp_scores=group['OppScore'].values,
            is_home=group['IsHome'].values,
            is_playoff=group['IsPlayoff'].values
        )
        group['pre_game_elo'] = elo_vals
        team_elo_list.append(group)
    team_games = pd.concat(team_elo_list, ignore_index=True)
    
    # 4. 匹配对手Elo（改用等值连接，避免merge_asof的排序问题）
    print("🔗 匹配对手Elo（等值连接）...")
    # 构建对手Elo表（包含每个球队每天比赛前的Elo）
    opp_elo = team_games[['Season', 'TeamID', 'DayNum', 'pre_game_elo']].copy()
    # 直接等值连接
    team_games = team_games.merge(
        opp_elo,
        left_on=['Season', 'OppTeamID', 'DayNum'],
        right_on=['Season', 'TeamID', 'DayNum'],
        how='left',
        suffixes=('', '_opp')
    )
    # 重命名合并后的 pre_game_elo_opp 为 opp_pre_game_elo
    team_games.rename(columns={'pre_game_elo_opp': 'opp_pre_game_elo'}, inplace=True)
    # 如果对手Elo缺失（理论上不会），填充默认值
    team_games['opp_pre_game_elo'] = team_games['opp_pre_game_elo'].fillna(1500.0).astype(np.float32)
    team_games['elo_diff'] = (team_games['pre_game_elo'] - team_games['opp_pre_game_elo']).astype(np.float32)
    
    del grouped_elo, team_elo_list, opp_elo
    gc.collect()
    
    # 5. 种子差特征（修复：仅锦标赛比赛可用，常规赛无种子数据）
    print("\n🌱 处理种子数据...")
    seeds_data['seed_num'] = seeds_data['Seed'].apply(extract_seed).astype(np.float32)
    # 只给锦标赛比赛添加种子特征，常规赛比赛种子特征为0
    team_games = team_games.merge(
        seeds_data[['Season', 'TeamID', 'seed_num', 'is_mens']], 
        on=['Season', 'TeamID', 'is_mens'], 
        how='left'
    ).rename(columns={'seed_num': 'team_seed'})
    # 常规赛比赛的种子特征设为0
    team_games['team_seed'] = team_games['team_seed'].where(team_games['IsPlayoff'].astype(bool), 0.0).astype(np.float32)
    
    # 对手种子特征
    team_games = team_games.merge(
        seeds_data[['Season', 'TeamID', 'seed_num', 'is_mens']],
        left_on=['Season', 'OppTeamID', 'is_mens'], 
        right_on=['Season', 'TeamID', 'is_mens'],
        how='left', 
        suffixes=('', '_opp')
    ).rename(columns={'seed_num': 'opp_seed'})
    # 常规赛比赛的对手种子特征设为0
    team_games['opp_seed'] = team_games['opp_seed'].where(team_games['IsPlayoff'].astype(bool), 0.0).astype(np.float32)
    team_games['seed_diff'] = (team_games['team_seed'] - team_games['opp_seed']).astype(np.float32)
    gc.collect()
    
    # 6. SOS与AdjEM特征
    print("\n📊 计算SOS和AdjEM特征...")
    team_games['team_win_flag'] = (team_games['Score'] > team_games['OppScore']).astype(np.int32)
    grouped_sos = team_games.groupby(['Season', 'TeamID'])
    
    sos_list = []
    for (season, team_id), group in tqdm(grouped_sos, desc="SOS计算进度", total=len(grouped_sos)):
        group = group.sort_values('DayNum')
        sos_vals = incremental_sos_calculation(
            day_nums=group['DayNum'].values,
            opp_team_ids=group['OppTeamID'].values,
            team_win_flags=group['team_win_flag'].values
        )
        sos_list.extend(sos_vals)
    
    team_games['sos'] = np.array(sos_list, dtype=np.float32)
    
    # 计算AdjEM
    print("📈 计算调整效率AdjEM...")
    team_games['possessions'] = (team_games['FGA'] - team_games['OR'] + team_games['TO'] + 0.44 * team_games['FTA']).astype(np.float32)
    team_games['opp_possessions'] = (team_games['OppFGA'] - team_games['OppOR'] + team_games['OppTO'] + 0.44 * team_games['OppFTA']).astype(np.float32)
    
    grouped_adj = team_games.groupby(['Season', 'TeamID'])
    cum_off_eff = grouped_adj.apply(
        lambda x: (x['Score'].shift(1).cumsum() / x['possessions'].shift(1).cumsum() * 100).fillna(0.0)
    ).reset_index(level=[0,1], drop=True).astype(np.float32)
    cum_def_eff = grouped_adj.apply(
        lambda x: (x['OppScore'].shift(1).cumsum() / x['opp_possessions'].shift(1).cumsum() * 100).fillna(0.0)
    ).reset_index(level=[0,1], drop=True).astype(np.float32)
    team_games['cum_off_eff'] = cum_off_eff
    team_games['cum_def_eff'] = cum_def_eff
    team_games['adj_em'] = (team_games['cum_off_eff'] - team_games['cum_def_eff']).astype(np.float32)
    
    del sos_list, grouped_sos, grouped_adj
    gc.collect()
    
    # 7. 数据泄露检查（修复：去除重复列）
    all_feature_cols = (
        rolling_cols + four_factor_cols +
        ['pre_game_elo', 'opp_pre_game_elo', 'elo_diff', 'team_seed', 'opp_seed', 'seed_diff', 'sos', 'adj_em']
    )
    # 去重：确保所有特征列唯一
    all_feature_cols = list(dict.fromkeys(all_feature_cols))
    leakage_guard = DataLeakageGuard(team_games)
    leakage_guard.validate_all_features(all_feature_cols)
    
    # 7. 整理最终特征矩阵（补上漏掉的关键列！）
    base_cols = ['Season', 'TeamID', 'OppTeamID', 'DayNum', 'is_mens', 'IsPlayoff', 'Score', 'OppScore']  # 加这三个！
    final_cols = base_cols + all_feature_cols
    # 再次去重，确保没有重复列
    final_cols = list(dict.fromkeys(final_cols))
    feature_matrix = team_games[final_cols].copy()
    
    # 最终内存优化
    int_cols = feature_matrix.select_dtypes(include=['int64']).columns
    float_cols = feature_matrix.select_dtypes(include=['float64']).columns
    feature_matrix[int_cols] = feature_matrix[int_cols].astype('int32')
    feature_matrix[float_cols] = feature_matrix[float_cols].astype('float32')
    
    # 最后检查列名唯一性
    assert feature_matrix.columns.is_unique, "特征矩阵存在重复列名"
    
    return feature_matrix

# ==================== 10. 主执行流程 ====================
if __name__ == "__main__":
    game_data, seeds_data = load_and_preprocess_data(BASE_DATA_PATH)
    feature_matrix = build_features_gpu(game_data, seeds_data)
    
    print(f"\n💾 保存特征矩阵至 {OUTPUT_PATH}...")
    # ── 保存前验证 ────────────────────────────────────────────────────────
    print(f"总样本量: {len(feature_matrix)}")
    print(f"IsPlayoff=True 的样本量: {feature_matrix['IsPlayoff'].sum()} （应接近 5800 行）")
    feature_matrix.to_parquet(OUTPUT_PATH, index=False)
    
    print("\n" + "="*80)
    print(f"✅ 特征矩阵已成功保存至 {OUTPUT_PATH}")
    print(f"📊 特征矩阵规模：{len(feature_matrix)}行 × {len(feature_matrix.columns)}列")
    print(f"🎯 所有特征严格遵循时序安全规则，无数据泄露，可直接传入Grok使用")
    print("="*80)

✅ 检测到GPU环境：可用GPU数量 2
  - GPU0: b'Tesla T4'
  - GPU1: b'Tesla T4'

🔍 检查数据文件是否存在...
✅ 找到文件: MRegularSeasonDetailedResults.csv
✅ 找到文件: WRegularSeasonDetailedResults.csv
✅ 找到文件: MNCAATourneyDetailedResults.csv
✅ 找到文件: WNCAATourneyDetailedResults.csv
✅ 找到文件: MNCAATourneySeeds.csv
✅ 找到文件: WNCAATourneySeeds.csv

📥 加载数据...
  常规赛行数: M=118882  W=81708
  锦标赛行数: M=1449  W=961
🔄 转换为球队视角数据（IsPlayoff 列随行传递）...
💾 优化数据类型...
🔍 检查并处理NaN值...
✅ 数据加载完成，总比赛行数：406000

⚙️ 开始构建全量特征（准确率优先模式）...


🏀 计算四大要素特征: 100%|██████████| 4/4 [00:01<00:00,  3.15it/s]



🧮 计算Elo评级（GPU加速）...


Elo计算进度: 100%|██████████| 13583/13583 [00:12<00:00, 1098.58it/s]


🔗 匹配对手Elo（等值连接）...

🌱 处理种子数据...

📊 计算SOS和AdjEM特征...


SOS计算进度: 100%|██████████| 13583/13583 [00:10<00:00, 1356.88it/s]


📈 计算调整效率AdjEM...
✅ 所有时序特征无数据泄露（合理初始值已验证）

💾 保存特征矩阵至 /kaggle/working/doubao_features.parquet...
总样本量: 406000
IsPlayoff=True 的样本量: 4820 （应接近 5800 行）

✅ 特征矩阵已成功保存至 /kaggle/working/doubao_features.parquet
📊 特征矩阵规模：406000行 × 51列
🎯 所有特征严格遵循时序安全规则，无数据泄露，可直接传入Grok使用


In [3]:
#!/usr/bin/env python3
# =============================================================================
# ncaa_2026_gold_medal_v11.py  (v11 — Top50特征 + LGB充分训练)
# NCAA March Machine Learning Mania 2026 — 金牌冲刺 v11
# =============================================================================
# v11 新增（基于feature_importance分析）:
#   A [★★] top_n 40→50：feature_importance显示第41-50名均有正向Gain(604K-669K)
#           关键：opp_pace_estimate(第46名M)被Top40卡掉，扩展后纳入
#           tempo_diff(第16名)、three_rate(第32名) 已在Top40，不受影响
#   B [★★] LGB final lr 0.02→0.015：best_iter=529在v8/v9/v10完全相同
#           2019早停点固定，降lr期望iter提升到700-900，充分训练全量数据
# 顺便发现：seed_hist_win_rate/tourney_past_final4/last10_score/last10_opp_score
#           在2003-2019预选LGB中重要性为0（早期赛季缺失），实际未被使用
#           last10_margin(第5名)已覆盖了recent form的核心信号
# v10保留: alpha自动判断(≥0.985跳过shrink), CAT depth=7/iter=5000/lr=0.03
# v9保留: OOF alpha搜索框架, 加权CV打印
# v8保留: tempo_diff, three_rate, T范围[0.87,1.15], clip[0.025,0.975]
# v7保留: 锦标赛×3.5, 2025权重2.8
# v6保留: P6-FIX1~P6-FIX6, BUG-1~BUG-5
# =============================================================================

import os
import gc
import math
import random
import warnings
from copy import deepcopy
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Any

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
import lightgbm as lgb
import xgboost as xgb

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
    print("✅ CatBoost 可用")
except Exception:
    HAS_CATBOOST = False
    print("⚠️ CatBoost 不可用，将用 ExtraTreesRegressor 作为替代")
    from sklearn.ensemble import ExtraTreesRegressor

warnings.filterwarnings('ignore')

# -----------------------
# Global config and paths
# -----------------------
BASE_PATH = '/kaggle/input/competitions/march-machine-learning-mania-2026/'
WORKING = '/kaggle/working/'

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# P5-FIX8: main submission temperature (T=1.10)
SUBMISSION_TEMPERATURE = 1.10

# P5-FIX3: remove composite_strength and relative_elo from GLM base feats
GLM_NEW_COLS = ['glm_team_quality', 'glm_opp_quality', 'glm_quality_diff']

# Required feature matrix (doubao)
FEATURE_PATH = f'{WORKING}doubao_features.parquet'
if not os.path.exists(FEATURE_PATH):
    raise FileNotFoundError(
        f"❌ 豆包特征文件不存在: {FEATURE_PATH}\n请先运行 doubao_features.py 生成特征矩阵。"
    )

# -----------------------
# Utilities & Validator
# -----------------------
@dataclass
class ValidationSplit:
    train_seasons: List[int]
    val_season: int
    train_data: pd.DataFrame
    val_data: pd.DataFrame


class TimeSeriesValidator:
    def __init__(self, min_train_seasons: int = 10, step: int = 1, purge_gap_seasons: int = 0):
        self.min_train_seasons = min_train_seasons
        self.step = step
        self.purge_gap_seasons = purge_gap_seasons

    def create_splits(self, all_seasons: List[int]) -> List[Tuple[List[int], int]]:
        valid = sorted(s for s in all_seasons if s != 2020)
        splits = []
        for i in range(self.min_train_seasons, len(valid), self.step):
            val_s = valid[i]
            train_ss = valid[:i - self.purge_gap_seasons] if self.purge_gap_seasons else valid[:i]
            if len(train_ss) >= self.min_train_seasons:
                splits.append((train_ss, val_s))
        return splits

    def split_dataframe(self, df: pd.DataFrame, season_col: str = 'Season') -> List[ValidationSplit]:
        all_seasons = sorted(df[season_col].unique().tolist())
        results = []
        for train_ss, val_s in self.create_splits(all_seasons):
            train_df = df[df[season_col].isin(train_ss)].copy()
            val_df = df[df[season_col] == val_s].copy()
            results.append(ValidationSplit(train_ss, val_s, train_df, val_df))
        return results


# -----------------------
# SimpleCalibrator (P5-FIX1)
# -----------------------
class SimpleCalibrator:
    """
    每折独立的简单分箱平滑校准器。
    n_bins=20, min_samples_per_bin=20, smoothing s = len(valid_bins) * 0.1
    """
    def __init__(self, n_bins: int = 20, min_samples: int = 20):
        self.n_bins = n_bins
        self.min_samples = min_samples
        self.bin_edges = None
        self.bin_means: Dict = {}
        self.global_mean = 0.5

    def fit(self, preds: np.ndarray, results: np.ndarray):
        df_tmp = pd.DataFrame({'x': preds, 'y': results})
        if len(df_tmp) == 0:
            self.global_mean = 0.5
            self.bin_edges = None
            self.bin_means = {}
            return self
        self.global_mean = float(df_tmp['y'].mean())
        xs = df_tmp['x'].values
        if np.all(xs == xs[0]):
            self.bin_edges = None
            self.bin_means = {}
            return self
        edges = np.linspace(xs.min(), xs.max(), self.n_bins + 1)
        df_tmp['bin'] = pd.cut(df_tmp['x'], bins=edges, include_lowest=True)
        bs = (df_tmp.groupby('bin', observed=True)['y']
              .agg(['mean', 'count']).reset_index())
        bs['mid'] = bs['bin'].apply(lambda b: b.mid if hasattr(b, 'mid') else np.nan)
        bs = bs.dropna(subset=['mid'])
        valid = bs[bs['count'] >= self.min_samples].copy()
        if valid.empty:
            self.bin_edges = None
            self.bin_means = {}
            return self
        s = max(1.0, len(valid) * 0.1)
        for _, row in valid.iterrows():
            b_mid = row['mid']
            cnt = float(row['count'])
            m = float(row['mean'])
            sm = (cnt * m + s * self.global_mean) / (cnt + s)
            self.bin_means[b_mid] = float(np.clip(sm, 1e-6, 1 - 1e-6))
        self.bin_edges = edges
        return self

    def predict(self, preds: np.ndarray) -> np.ndarray:
        if self.bin_edges is None or len(self.bin_means) == 0:
            return np.full(len(preds), fill_value=float(np.clip(self.global_mean, 1e-6, 1 - 1e-6)))
        bins = np.digitize(preds, self.bin_edges) - 1
        out = []
        mids = np.array(list(self.bin_means.keys()))
        means = np.array(list(self.bin_means.values()))
        for p, b in zip(preds, bins):
            if b < 0 or b >= (len(self.bin_edges) - 1):
                out.append(self.global_mean)
                continue
            left = self.bin_edges[b]
            right = self.bin_edges[b + 1]
            mid = (left + right) / 2.0
            if mid in self.bin_means:
                out.append(self.bin_means[mid])
            else:
                idx = int((np.abs(mids - mid)).argmin())
                out.append(means[idx])
        return np.clip(np.array(out, dtype=float), 1e-6, 1 - 1e-6)


# -----------------------
# Feature engineering
# -----------------------
def build_conference_strength(base_path: str) -> pd.DataFrame:
    conf_dfs = []
    for prefix, fname in [('M', 'MTeamConferences.csv'), ('W', 'WTeamConferences.csv')]:
        fpath = os.path.join(base_path, fname)
        if not os.path.exists(fpath):
            continue
        cf = pd.read_csv(fpath)
        cf['is_mens'] = 1 if prefix == 'M' else 0
        conf_dfs.append(cf[['Season', 'TeamID', 'ConfAbbrev', 'is_mens']])
    if not conf_dfs:
        return pd.DataFrame(columns=['Season', 'TeamID', 'conf_strength', 'conf_rank'])

    conf = pd.concat(conf_dfs, ignore_index=True)
    result_dfs = []
    for prefix, fname in [('M', 'MRegularSeasonDetailedResults.csv'),
                           ('W', 'WRegularSeasonDetailedResults.csv')]:
        fpath = os.path.join(base_path, fname)
        if not os.path.exists(fpath):
            continue
        r = pd.read_csv(fpath)[['Season', 'WTeamID', 'LTeamID']]
        r['is_mens'] = 1 if prefix == 'M' else 0
        result_dfs.append(r)

    if not result_dfs:
        return pd.DataFrame(columns=['Season', 'TeamID', 'conf_strength', 'conf_rank'])

    results = pd.concat(result_dfs, ignore_index=True)
    wins = results.groupby(['Season', 'WTeamID'])['LTeamID'].count().reset_index()
    wins.columns = ['Season', 'TeamID', 'wins']
    losses = results.groupby(['Season', 'LTeamID'])['WTeamID'].count().reset_index()
    losses.columns = ['Season', 'TeamID', 'losses']
    wl = wins.merge(losses, on=['Season', 'TeamID'], how='outer').fillna(0)
    wl['win_rate'] = wl['wins'] / (wl['wins'] + wl['losses'] + 1e-9)

    team_conf = conf.merge(wl, on=['Season', 'TeamID'], how='left').fillna(0)
    conf_strength = (team_conf.groupby(['Season', 'ConfAbbrev'])['win_rate']
                     .mean().reset_index()
                     .rename(columns={'win_rate': 'conf_strength'}))
    conf_strength['conf_rank'] = (conf_strength.groupby('Season')['conf_strength']
                                  .rank(ascending=False, method='dense'))
    conf_strength = conf_strength.sort_values('Season')
    conf_strength['conf_strength_lag'] = conf_strength.groupby('ConfAbbrev')['conf_strength'].shift(1)
    conf_strength['conf_rank_lag'] = conf_strength.groupby('ConfAbbrev')['conf_rank'].shift(1)
    conf_strength['conf_strength'] = conf_strength['conf_strength_lag'].fillna(0.5)
    conf_strength['conf_rank'] = conf_strength['conf_rank_lag'].fillna(15.0)

    team_conf = team_conf.merge(
        conf_strength[['Season', 'ConfAbbrev', 'conf_strength', 'conf_rank']],
        on=['Season', 'ConfAbbrev'], how='left'
    )
    return team_conf[['Season', 'TeamID', 'conf_strength', 'conf_rank']].fillna(0.5)


def add_feature_interactions(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    new_cols = []
    if all(c in df.columns for c in ['rolling_10game_mean_FGA', 'rolling_10game_mean_OR',
                                      'rolling_10game_mean_TO', 'rolling_10game_mean_FTA']):
        df['pace_estimate'] = (
            df['rolling_10game_mean_FGA'] - df['rolling_10game_mean_OR']
            + df['rolling_10game_mean_TO'] + 0.44 * df['rolling_10game_mean_FTA']
        ).astype('float32')
        new_cols.append('pace_estimate')

    if 'rolling_10game_mean_eFG_pct' in df.columns:
        df['efg_advantage'] = (df['rolling_10game_mean_eFG_pct'] - 0.50).astype('float32')
        new_cols.append('efg_advantage')

    if all(c in df.columns for c in ['rolling_10game_mean_eFG_pct', 'rolling_10game_mean_Score']):
        df['efg_score_cross'] = (
            (df['rolling_10game_mean_eFG_pct'] - 0.50) * df['rolling_10game_mean_Score'] / 70.0
        ).astype('float32')
        new_cols.append('efg_score_cross')

    if all(c in df.columns for c in ['rolling_10game_mean_TOV_pct', 'rolling_10game_mean_TO']):
        df['tov_advantage'] = (-df['rolling_10game_mean_TOV_pct']).astype('float32')
        new_cols.append('tov_advantage')

    if 'rolling_10game_mean_ORB_pct' in df.columns:
        df['orb_advantage'] = df['rolling_10game_mean_ORB_pct'].astype('float32')
        new_cols.append('orb_advantage')

    if 'rolling_10game_mean_FT_rate' in df.columns:
        df['ftr_advantage'] = df['rolling_10game_mean_FT_rate'].astype('float32')
        new_cols.append('ftr_advantage')

    if 'seed_diff' in df.columns and 'elo_diff' in df.columns:
        df['seed_elo_interaction'] = (df['seed_diff'] * df['elo_diff'] / 1000.0).astype('float32')
        df['elo_diff_sq'] = (df['elo_diff'] ** 2 / 10000.0).astype('float32')
        df['seed_diff_sq'] = (df['seed_diff'] ** 2).astype('float32')
        new_cols += ['seed_elo_interaction', 'elo_diff_sq', 'seed_diff_sq']

    if all(c in df.columns for c in ['rolling_5game_mean_Score', 'rolling_10game_mean_Score']):
        df['score_momentum'] = (
            df['rolling_5game_mean_Score'] - df['rolling_10game_mean_Score']
        ).astype('float32')
        new_cols.append('score_momentum')

    if all(c in df.columns for c in ['rolling_5game_mean_OppScore', 'rolling_10game_mean_OppScore']):
        df['def_momentum'] = (
            df['rolling_10game_mean_OppScore'] - df['rolling_5game_mean_OppScore']
        ).astype('float32')
        new_cols.append('def_momentum')

    if 'pre_game_elo' in df.columns and 'opp_pre_game_elo' in df.columns:
        df['elo_sum'] = ((df['pre_game_elo'] + df['opp_pre_game_elo']) / 2000.0).astype('float32')
        new_cols.append('elo_sum')

    if 'adj_em' in df.columns:
        df['adj_em_sq'] = (df['adj_em'] ** 2 / 100.0).astype('float32')
        new_cols.append('adj_em_sq')

    # ChatGPT GM建议1: tempo_diff matchup（style-clash特征，快节奏vs慢节奏更易爆冷）
    # pace_estimate 已有，补充对手 pace 并计算差值
    if 'pace_estimate' in df.columns:
        opp_pace_cols = ['rolling_10game_mean_OppFGA', 'rolling_10game_mean_OppOR',
                         'rolling_10game_mean_OppTO', 'rolling_10game_mean_OppFTA']
        if all(c in df.columns for c in opp_pace_cols):
            df['opp_pace_estimate'] = (
                df['rolling_10game_mean_OppFGA'] - df['rolling_10game_mean_OppOR']
                + df['rolling_10game_mean_OppTO'] + 0.44 * df['rolling_10game_mean_OppFTA']
            ).astype('float32')
            df['tempo_diff'] = (df['pace_estimate'] - df['opp_pace_estimate']).astype('float32')
            new_cols += ['opp_pace_estimate', 'tempo_diff']

    # ChatGPT GM建议2: three_rate_diff（高三分出手队方差大→更易爆冷，对Brier很重要）
    if all(c in df.columns for c in ['rolling_10game_mean_FGM3', 'rolling_10game_mean_FGA',
                                      'rolling_10game_mean_FGM', 'rolling_10game_mean_FGA3']):
        df['three_rate'] = (
            df['rolling_10game_mean_FGA3'] / (df['rolling_10game_mean_FGA'] + 1e-6)
        ).astype('float32')
        new_cols.append('three_rate')

    print(f"  ✅ 特征工程 2.0 新增 {len(new_cols)} 个特征 (P5-FIX3 applied)")
    return df, new_cols


# -----------------------
# P6-FIX2: 历史锦标赛表现特征 (豆包 #1)
# -----------------------
def build_tourney_history_features(base_path: str) -> pd.DataFrame:
    """
    P6-FIX2 (稳健版):
      tourney_past_wins   : 过去5年锦标赛胜场 rolling(5).sum().shift(1)
      tourney_past_final4 : 过去5年是否进过 Final Four rolling(5).max().shift(1)
      Final Four 定义: 同一赛季锦标赛胜场 >= 4

    稳健性改进:
      - 同时尝试 Results.csv 和 CompactResults.csv（任意一个即可）
      - merge 后立即 fillna(0)，杜绝 t_wins NaN
      - rolling/shift 后再次 fillna(0)（保护第一行 shift 产生的 NaN）
      - dropna(subset=['Season','TeamID']) 防止空键入库
      - 最终返回前再做全列 fillna(0) + astype 双重保险
    全部 shift(1) 防泄露。
    """
    tourney_dfs = []
    for fname in ['MNCAATourneyResults.csv', 'WNCAATourneyResults.csv',
                  'MNCAATourneyCompactResults.csv', 'WNCAATourneyCompactResults.csv']:
        fpath = os.path.join(base_path, fname)
        if os.path.exists(fpath):
            try:
                r = pd.read_csv(fpath)
                # 只保留必需列（Results 和 CompactResults 均包含）
                needed = {'Season', 'WTeamID', 'LTeamID'}
                if needed.issubset(r.columns):
                    tourney_dfs.append(r[list(needed)])
            except Exception as e:
                print(f"  ⚠️ P6-FIX2: 读取 {fname} 出错: {e}")

    if not tourney_dfs:
        print("  ⚠️ P6-FIX2: 未找到任何锦标赛文件，跳过历史特征")
        return pd.DataFrame(columns=['Season', 'TeamID', 'tourney_past_wins', 'tourney_past_final4'])

    # 合并去重（多个文件可能有重叠赛季）
    tourney = (pd.concat(tourney_dfs, ignore_index=True)
               .drop_duplicates(subset=['Season', 'WTeamID', 'LTeamID'])
               .dropna(subset=['Season', 'WTeamID', 'LTeamID'])
               .reset_index(drop=True))
    tourney['Season']   = tourney['Season'].astype(int)
    tourney['WTeamID']  = tourney['WTeamID'].astype(int)
    tourney['LTeamID']  = tourney['LTeamID'].astype(int)

    # 胜场数 per (Season, WTeamID)
    wins_per = (tourney.groupby(['Season', 'WTeamID'])
                .size().reset_index(name='t_wins')
                .rename(columns={'WTeamID': 'TeamID'}))

    # 所有参赛队（胜者 ∪ 败者）→ 败者胜场为 0
    w_teams = wins_per[['Season', 'TeamID']].copy()
    l_teams = (tourney[['Season', 'LTeamID']]
               .rename(columns={'LTeamID': 'TeamID'})
               .drop_duplicates())
    all_ts = (pd.concat([w_teams, l_teams], ignore_index=True)
              .drop_duplicates(subset=['Season', 'TeamID'])
              .dropna(subset=['Season', 'TeamID'])
              .reset_index(drop=True))

    # 合并胜场；fillna(0) 确保败者/未参赛者胜场为 0
    team_wins = all_ts.merge(wins_per, on=['Season', 'TeamID'], how='left')
    team_wins['t_wins']   = team_wins['t_wins'].fillna(0).astype('float32')
    team_wins['t_final4'] = (team_wins['t_wins'] >= 4).astype('float32')

    # 按 (TeamID, Season) 升序排序后做 rolling shift(1)
    team_wins = team_wins.sort_values(['TeamID', 'Season']).reset_index(drop=True)

    team_wins['tourney_past_wins'] = (
        team_wins.groupby('TeamID', sort=False)['t_wins']
        .transform(lambda x: x.rolling(5, min_periods=1).sum().shift(1))
        .fillna(0)                   # 第一行 shift(1) → NaN → 填 0
        .astype('float32')
    )

    team_wins['tourney_past_final4'] = (
        team_wins.groupby('TeamID', sort=False)['t_final4']
        .transform(lambda x: x.rolling(5, min_periods=1).max().shift(1))
        .fillna(0)
        .astype('float32')
    )

    result = (team_wins[['Season', 'TeamID', 'tourney_past_wins', 'tourney_past_final4']]
              .fillna(0)              # 最终兜底 fillna
              .reset_index(drop=True))
    print(f"  ✅ P6-FIX2 (稳健版): 历史锦标赛特征构建完成，{len(result)} 个队季度记录")
    return result


# -----------------------
# P6-FIX3: 种子历史胜率字典 (豆包 #2)
# seed_diff = team_seed - opp_seed
# 完整字典: seed_diff in [-15, 15]
# -----------------------
SEED_WIN_RATE_DICT = {
    -15: 0.013,
    -14: 0.062,
    -13: 0.085,
    -12: 0.118,
    -11: 0.150,
    -10: 0.185,
    -9:  0.220,
    -8:  0.260,
    -7:  0.300,
    -6:  0.350,
    -5:  0.392,
    -4:  0.425,
    -3:  0.460,
    -2:  0.480,
    -1:  0.495,
     0:  0.500,
     1:  0.505,
     2:  0.520,
     3:  0.540,
     4:  0.575,
     5:  0.608,
     6:  0.650,
     7:  0.700,
     8:  0.740,
     9:  0.780,
    10:  0.815,
    11:  0.850,
    12:  0.882,
    13:  0.915,
    14:  0.938,
    15:  0.987,
}


# -----------------------
# GLM features (P5-FIX4: Ridge on PointDiff)
# -----------------------
def add_glm_features_to_fold(train_df: pd.DataFrame, val_df: pd.DataFrame,
                             static_feat_cols: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train = train_df.copy().reset_index(drop=True)
    val = val_df.copy().reset_index(drop=True)
    global_mean = 0.0

    avail = [c for c in static_feat_cols if c in train.columns and c in val.columns]

    if len(avail) < 3 or len(train) < 200:
        for col in GLM_NEW_COLS:
            train[col] = global_mean
            val[col] = global_mean
        return train, val, pd.DataFrame(columns=['Season', 'TeamID', 'glm_quality'])

    def build_X(df):
        rows = []
        for _, r in df.iterrows():
            tv, ov = [], []
            for c in avail:
                tv.append(r[c] if pd.notna(r[c]) else 0.0)
                opp_c = f'opp_{c}'
                ov.append(r[opp_c] if opp_c in df.columns and pd.notna(r[opp_c]) else 0.0)
            rows.append(np.array(tv) - np.array(ov))
        return np.vstack(rows).astype(float)

    X_tr = build_X(train)
    y_tr = train['PointDiff'].values.astype(float)
    ridge = Ridge(alpha=1.0, random_state=SEED)
    ridge.fit(X_tr, y_tr)
    pred_tr = ridge.predict(X_tr)

    tmp = pd.DataFrame({'Season': train['Season'].values, 'TeamID': train['TeamID'].values,
                        'pred_margin': pred_tr})
    quality_map = (tmp.groupby(['Season', 'TeamID'])['pred_margin'].mean()
                   .reset_index().rename(columns={'pred_margin': 'glm_quality'}))
    global_mean = float(quality_map['glm_quality'].mean()) if not quality_map.empty else 0.0

    q_lagged = quality_map.copy()
    q_lagged['Season'] = q_lagged['Season'] + 1
    q_dict = q_lagged.set_index(['Season', 'TeamID'])['glm_quality'].to_dict()

    train['glm_team_quality'] = [q_dict.get((int(s), int(t)), global_mean)
                                 for s, t in zip(train['Season'], train['TeamID'])]
    train['glm_opp_quality'] = [q_dict.get((int(s), int(t)), global_mean)
                                for s, t in zip(train['Season'], train['OppTeamID'])]
    train['glm_quality_diff'] = (train['glm_team_quality'] - train['glm_opp_quality']).astype('float32')

    last_s = int(train['Season'].max())
    last_q_dict = quality_map[quality_map['Season'] == last_s].set_index('TeamID')['glm_quality'].to_dict()
    val['glm_team_quality'] = [last_q_dict.get(int(t), global_mean) for t in val['TeamID']]
    val['glm_opp_quality'] = [last_q_dict.get(int(t), global_mean) for t in val['OppTeamID']]
    val['glm_quality_diff'] = (val['glm_team_quality'] - val['glm_opp_quality']).astype('float32')

    return train, val, quality_map


def build_final_glm_quality_map(df_all: pd.DataFrame,
                                static_feat_cols: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df = df_all.copy().reset_index(drop=True)
    avail = [c for c in static_feat_cols if c in df.columns]

    if len(avail) < 3 or len(df) < 200:
        for col in GLM_NEW_COLS:
            df[col] = 0.0
        return df, pd.DataFrame(columns=['Season', 'TeamID', 'glm_quality'])

    def build_X(df_local):
        rows = []
        for _, r in df_local.iterrows():
            tv, ov = [], []
            for c in avail:
                tv.append(r[c] if pd.notna(r[c]) else 0.0)
                opp_c = f'opp_{c}'
                ov.append(r[opp_c] if opp_c in df_local.columns and pd.notna(r[opp_c]) else 0.0)
            rows.append(np.array(tv) - np.array(ov))
        return np.vstack(rows).astype(float)

    X_all = build_X(df)
    y_all = df['PointDiff'].values.astype(float)
    ridge = Ridge(alpha=1.0, random_state=SEED)
    ridge.fit(X_all, y_all)
    preds = ridge.predict(X_all)

    tmp = pd.DataFrame({'Season': df['Season'].values, 'TeamID': df['TeamID'].values,
                        'pred_margin': preds})
    quality_map = (tmp.groupby(['Season', 'TeamID'])['pred_margin'].mean()
                   .reset_index().rename(columns={'pred_margin': 'glm_quality'}))

    q_lagged = quality_map.copy()
    q_lagged['Season'] = q_lagged['Season'] + 1
    q_dict = q_lagged.set_index(['Season', 'TeamID'])['glm_quality'].to_dict()
    global_mean = float(quality_map['glm_quality'].mean()) if not quality_map.empty else 0.0

    df['glm_team_quality'] = [q_dict.get((int(s), int(t)), global_mean)
                              for s, t in zip(df['Season'], df['TeamID'])]
    df['glm_opp_quality'] = [q_dict.get((int(s), int(t)), global_mean)
                             for s, t in zip(df['Season'], df['OppTeamID'])]
    df['glm_quality_diff'] = (df['glm_team_quality'] - df['glm_opp_quality']).astype('float32')

    print(f"  ✅ 全量GLM质量图已构建，覆盖 {len(quality_map)} 个队季度 (P5-FIX4)")
    return df, quality_map


# -----------------------
# Sample weights (v7: stepped time weights + playoff ×3.5)
# 修改说明:
#   1. 锦标赛权重 ×2.0 → ×3.5：让模型更专注学习锦标赛模式
#   2. 仅提升 2025/2024 时间权重（不动 else=1.0），近2年有效占比 21.4% → 24.9%
#      注意：豆包/grok 把 else 也从 1.0→1.2，反而会稀释近年占比（21.4%→20.8%），效果相反
# -----------------------
def _make_sample_weights(seasons: np.ndarray, is_playoff: np.ndarray = None) -> np.ndarray:
    w = np.ones(len(seasons), dtype='float32')
    w[np.where(seasons >= 2025)] = 2.8   # 原2.2 → 2.8（仅提升近两年）
    w[np.where(seasons == 2024)] = 2.2   # 原1.9 → 2.2
    w[np.where(seasons == 2023)] = 1.6   # 保持不变
    w[np.where(seasons == 2022)] = 1.3   # 保持不变
    # else (≤2021): 保持 1.0，不动——防止稀释近年占比
    if is_playoff is not None:
        w[is_playoff] *= 3.5             # 原×2.0 → ×3.5
    return w


# -----------------------
# Model builder
# -----------------------
def _build_catboost_model(best_params: Dict) -> Any:
    if HAS_CATBOOST:
        # P2(v9): iterations 1500→5000, depth 6→8, lr 0.05→0.03, early_stopping 60→120
        # CAT权重持续主导(v6:0.39→v7:0.46→v8:0.44)，值得加深；lr降低配合更多轮次防过拟合
        # v10修正: depth 8→7（v9 depth=8在CV fold略微过拟合，CAT权重从0.46降到0.37）
        return CatBoostRegressor(
            iterations=best_params.get('cb_iter', 5000),
            learning_rate=best_params.get('cb_lr', 0.03),
            depth=best_params.get('cb_depth', 7),
            l2_leaf_reg=best_params.get('cb_l2', 3.0),
            random_seed=SEED, verbose=0,
            loss_function='RMSE', eval_metric='RMSE',
            early_stopping_rounds=120,
        )
    else:
        return ExtraTreesRegressor(n_estimators=500, max_depth=10, min_samples_leaf=10,
                                   random_state=SEED, n_jobs=-1)


# -----------------------
# Dynamic feature selection (P6-FIX5: save _v6.csv, Top50)
# -----------------------
def select_top_features_dynamic(df: pd.DataFrame, gender_tag: str, top_n: int = 40) -> List[str]:
    """在 2003-2019 数据上训练 LGB，返回 Gain Top top_n 特征 (v11: Top50)"""
    train_df = df[df['Season'] <= 2019].copy()
    exclude = {'Season', 'DayNum', 'TeamID', 'OppTeamID', 'PointDiff', 'Result',
               'is_mens', 'Score', 'OppScore', 'Stl', 'Blk', 'PF',
               'OppStl', 'OppBlk', 'OppPF', 'WLoc', 'IsHome', 'IsPlayoff'}
    exclude.update(GLM_NEW_COLS)

    if len(train_df) == 0:
        candidates = [c for c in df.columns if c not in exclude
                      and df[c].dtype in ['float32', 'float64', 'int32', 'int64']]
        return candidates[:top_n]

    all_feat = [c for c in train_df.columns if c not in exclude
                and train_df[c].dtype in ['float32', 'float64', 'int32', 'int64']]
    if len(all_feat) == 0:
        return []

    X = train_df[all_feat].fillna(0)
    y = train_df['PointDiff']

    params = {
        'objective': 'regression', 'metric': 'rmse',
        'learning_rate': 0.05, 'num_leaves': 64,
        'feature_fraction': 0.8, 'bagging_fraction': 0.8,
        'bagging_freq': 5, 'verbose': -1, 'random_state': SEED
    }
    model = lgb.train(params, lgb.Dataset(X, label=y), num_boost_round=500)

    imp = pd.DataFrame({'feature': all_feat,
                        'gain': model.feature_importance(importance_type='gain')})
    imp = imp.sort_values('gain', ascending=False).reset_index(drop=True)
    top_feats = imp.head(top_n)['feature'].tolist()
    print(f"\n  [{gender_tag}] 动态筛选 Top{top_n} 特征完成，前5: {top_feats[:5]}")

    # P6-FIX5: 保存特征重要性报告 Top50 → _v6.csv
    imp.head(50).to_csv(f'{WORKING}feature_importance_{gender_tag}_v6.csv', index=False)
    print(f"  [{gender_tag}] 特征重要性已保存 → feature_importance_{gender_tag}_v6.csv")

    return top_feats


# -----------------------
# Training pipeline per gender
# -----------------------
def train_gender_pipeline(df: pd.DataFrame,
                          feature_cols: List[str],
                          gender_tag: str,
                          best_lgb_params: Dict,
                          best_xgb_params: Dict,
                          n_optuna_trials: int = 0,
                          val_seasons: List[int] = None) -> Dict:
    """
    P6-FIX1: Final 全量模型用 2019 验证集做 early stopping (LGB/XGB 各 100 轮耐心)
    其余逻辑与 v5 保持一致，不引入 Stacking/Optuna/ThreeLevelCalibrator
    """
    if val_seasons is None:
        val_seasons = [2021, 2022, 2023, 2024, 2025]
    print(f"\n{'='*70}\n{('男子' if gender_tag == 'M' else '女子')} [{gender_tag}] training v6\n{'='*70}")

    static_feat_cols = [f for f in feature_cols if f not in GLM_NEW_COLS]
    validator = TimeSeriesValidator(min_train_seasons=10)
    splits = validator.split_dataframe(df)

    cv_results = {'xgb': [], 'lgb': [], 'cat': [], 'ensemble': []}
    oof_records = []
    lgb_best_iters = []

    lgb_params = {
        'objective': 'regression', 'metric': 'rmse',
        'verbose': -1, 'random_state': SEED,
        'learning_rate': best_lgb_params.get('lr', 0.04),
        'num_leaves': best_lgb_params.get('num_leaves', 64),
        'max_depth': best_lgb_params.get('max_depth', 7),
        'feature_fraction': best_lgb_params.get('ff', 0.85),
        'bagging_fraction': best_lgb_params.get('bf', 0.80),
        'bagging_freq': best_lgb_params.get('bag_freq', 5),
        'min_child_samples': best_lgb_params.get('min_child', 20),
        'lambda_l1': best_lgb_params.get('l1', 0.01),
        'lambda_l2': best_lgb_params.get('l2', 0.01),
        'max_bin': best_lgb_params.get('max_bin', 255),
    }
    xgb_cv_params = dict(
        objective='reg:squarederror', eval_metric='rmse', verbosity=0,
        random_state=SEED,
        learning_rate=best_xgb_params.get('lr', 0.01),
        max_depth=best_xgb_params.get('max_depth', 6),
        subsample=best_xgb_params.get('subsample', 0.65),
        colsample_bytree=best_xgb_params.get('colsample_bytree', 0.8),
        colsample_bylevel=best_xgb_params.get('colsample_bylevel', 0.8),
        min_child_weight=best_xgb_params.get('min_child_weight', 5),
        reg_alpha=best_xgb_params.get('alpha', 0.01),
        reg_lambda=best_xgb_params.get('lambda_', 1.0),
        gamma=best_xgb_params.get('gamma', 0.01),
        n_estimators=2000,
        early_stopping_rounds=60,
    )

    # ---- CV loop ----
    for split in splits:
        vs = split.val_season
        if vs not in val_seasons:
            continue

        fold_train, fold_val, _ = add_glm_features_to_fold(
            split.train_data, split.val_data, static_feat_cols)
        fold_feat_cols = static_feat_cols + [c for c in GLM_NEW_COLS if c in fold_train.columns]

        X_tr = fold_train[fold_feat_cols].fillna(0)
        y_tr = fold_train['PointDiff'].values.astype(float)
        X_va = fold_val[fold_feat_cols].fillna(0)
        y_va_pt = fold_val['PointDiff'].values.astype(float)
        y_va_res = fold_val['Result'].values.astype(int)

        fold_is_playoff = fold_train.get('IsPlayoff', pd.Series(0, index=fold_train.index)).values.astype(bool)
        sample_w = _make_sample_weights(fold_train['Season'].values, fold_is_playoff)

        fold_cal = SimpleCalibrator(n_bins=20, min_samples=20)
        fold_cal.fit(y_tr, fold_train['Result'].values)

        # XGB CV fold — BUG-2 FIX: 加 early_stopping_rounds 进构造函数（原代码排除了它但fit也没传，跑满2000轮）
        xgb_cv_params_clean = {k: v for k, v in xgb_cv_params.items()
                               if k not in ('early_stopping_rounds', 'n_estimators')}
        xgb_mdl = xgb.XGBRegressor(**xgb_cv_params_clean,
                                    n_estimators=xgb_cv_params.get('n_estimators', 2000),
                                    early_stopping_rounds=xgb_cv_params.get('early_stopping_rounds', 60))
        xgb_mdl.fit(X_tr, y_tr, sample_weight=sample_w,
                    eval_set=[(X_va, y_va_pt)], verbose=False)
        pred_xgb = xgb_mdl.predict(X_va)
        prob_xgb = fold_cal.predict(pred_xgb)

        # LGB CV fold
        tr_set = lgb.Dataset(X_tr, label=y_tr, weight=sample_w)
        va_set = lgb.Dataset(X_va, label=y_va_pt, reference=tr_set)
        lgb_mdl = lgb.train(lgb_params, tr_set, num_boost_round=2000,
                            valid_sets=[va_set],
                            callbacks=[lgb.early_stopping(80, verbose=False)])
        lgb_best_iters.append(lgb_mdl.best_iteration or 1500)
        pred_lgb = lgb_mdl.predict(X_va)
        prob_lgb = fold_cal.predict(pred_lgb)

        # CatBoost / ExtraTrees CV fold
        cat_mdl = _build_catboost_model({})
        if HAS_CATBOOST:
            cat_mdl.fit(X_tr, y_tr, sample_weight=sample_w,
                        eval_set=(X_va, y_va_pt), verbose=False)
        else:
            cat_mdl.fit(X_tr, y_tr, sample_weight=sample_w)
        pred_cat = cat_mdl.predict(X_va)
        prob_cat = fold_cal.predict(pred_cat)

        prob_ens = (prob_xgb + prob_lgb + prob_cat) / 3.0
        b_xgb = brier_score_loss(y_va_res, prob_xgb)
        b_lgb = brier_score_loss(y_va_res, prob_lgb)
        b_cat = brier_score_loss(y_va_res, prob_cat)
        b_ens = brier_score_loss(y_va_res, prob_ens)
        print(f"  [{gender_tag}] Season {vs}: XGB={b_xgb:.5f} LGB={b_lgb:.5f} CAT={b_cat:.5f} Ens={b_ens:.5f}")

        cv_results['xgb'].append(b_xgb)
        cv_results['lgb'].append(b_lgb)
        cv_results['cat'].append(b_cat)
        cv_results['ensemble'].append(b_ens)

        val_ids = fold_val[['Season', 'DayNum', 'TeamID', 'OppTeamID']].copy()
        val_ids['oof_xgb'] = pred_xgb
        val_ids['oof_lgb'] = pred_lgb
        val_ids['oof_cat'] = pred_cat
        val_ids['Result'] = y_va_res
        oof_records.append(val_ids)

    # ---- Bayesian weight optimization from OOF ----
    mean_briers = {k: np.mean(v) if v else 1.0 for k, v in cv_results.items() if k != 'ensemble'}
    inv_b = {k: 1.0 / (v + 1e-9) for k, v in mean_briers.items()}
    total = sum(inv_b.values())
    inv_brier_weights = {k: float(v / total) for k, v in inv_b.items()}
    opt_weights = inv_brier_weights.copy()

    best_T = SUBMISSION_TEMPERATURE
    if oof_records:
        oof_df = pd.concat(oof_records, ignore_index=True)
        oof_avg = (oof_df['oof_xgb'].values + oof_df['oof_lgb'].values + oof_df['oof_cat'].values) / 3.0
        oof_cal = SimpleCalibrator(n_bins=20, min_samples=20)
        oof_cal.fit(oof_avg, oof_df['Result'].values)
        p_xgb_oof = oof_cal.predict(oof_df['oof_xgb'].values)
        p_lgb_oof = oof_cal.predict(oof_df['oof_lgb'].values)
        p_cat_oof = oof_cal.predict(oof_df['oof_cat'].values)

        def brier_weights(w):
            w = np.abs(w)
            if w.sum() == 0:
                w = np.array([1/3, 1/3, 1/3])
            w = w / w.sum()
            p = w[0]*p_xgb_oof + w[1]*p_lgb_oof + w[2]*p_cat_oof
            return brier_score_loss(oof_df['Result'].values, np.clip(p, 0.03, 0.97))

        from scipy.optimize import minimize
        res = minimize(brier_weights, x0=np.array([1/3, 1/3, 1/3]),
                       method='Nelder-Mead', options={'maxiter': 500, 'xatol': 1e-5})
        raw_w = np.abs(res.x)
        raw_w = raw_w / (raw_w.sum() + 1e-12)
        opt_weights = {'xgb': float(raw_w[0]), 'lgb': float(raw_w[1]), 'cat': float(raw_w[2])}
        print(f"\n  [{gender_tag}] 贝叶斯最优集成权重: "
              f"XGB={opt_weights['xgb']:.3f} LGB={opt_weights['lgb']:.3f} CAT={opt_weights['cat']:.3f}")

        # P6-FIX6: OOF 动态温度优化
        # 收窄范围到 [0.87, 1.15]，防止优化器跑偏；当前实测T≈0.98已在范围内
        def optimize_temperature(oof_probs: np.ndarray, oof_true: np.ndarray) -> float:
            from scipy.optimize import minimize_scalar
            def neg_brier(log_t):
                T = np.exp(log_t)
                p = 1 / (1 + np.exp(-np.log(oof_probs / (1 - oof_probs + 1e-9)) / T))
                return brier_score_loss(oof_true, np.clip(p, 0.03, 0.97))
            res2 = minimize_scalar(neg_brier,
                                   bounds=(np.log(0.87), np.log(1.15)),
                                   method='bounded')
            return float(np.exp(res2.x))

        oof_ens = (p_xgb_oof + p_lgb_oof + p_cat_oof) / 3.0
        best_T = optimize_temperature(oof_ens, oof_df['Result'].values)
        print(f"  [{gender_tag}] OOF 优化最佳温度 T = {best_T:.3f}")

        # P1(v9): 基于温度后OOF概率搜索最优 shrink alpha ∈ [0.90, 0.99]
        # 原理：温度缩放后再搜索alpha，两步解耦，互不干扰
        def _apply_temp(p: np.ndarray, T: float) -> np.ndarray:
            eps = 1e-9
            logits = np.log(np.clip(p, eps, 1-eps) / (1 - np.clip(p, eps, 1-eps)))
            return np.clip(1.0 / (1.0 + np.exp(-logits / T)), 1e-6, 1-1e-6)

        def optimize_alpha(oof_probs_scaled: np.ndarray, oof_true: np.ndarray) -> float:
            from scipy.optimize import minimize_scalar
            def neg_brier_alpha(alpha):
                p = np.clip(0.5 + (oof_probs_scaled - 0.5) * alpha, 1e-6, 1-1e-6)
                return brier_score_loss(oof_true, np.clip(p, 0.03, 0.97))
            # v10: 上界扩展到1.00（v9上界0.99被命中，说明优化器想要更高）
            # alpha=1.00 = 恒等变换，即不做任何收缩
            res_a = minimize_scalar(neg_brier_alpha, bounds=(0.85, 1.00), method='bounded')
            return float(res_a.x)

        oof_ens_scaled = _apply_temp(oof_ens, best_T)
        best_alpha = optimize_alpha(oof_ens_scaled, oof_df['Result'].values)
        # 自动判断：alpha>0.985 视为"不需要收缩"，直接置1.00
        if best_alpha >= 0.985:
            print(f"  [{gender_tag}] OOF alpha={best_alpha:.4f} ≥ 0.985 → 模型校准良好，跳过shrink (alpha=1.00)")
            best_alpha = 1.00
        else:
            print(f"  [{gender_tag}] OOF 优化最佳 shrink alpha = {best_alpha:.4f}")

        # P3(v9): 加权CV汇报（2021权重最低，2025最高，反映对2026的真实参考价值）
        # 注意：这只改变"评估指标的打印方式"，不影响训练或提交
        _cv_w = {2021: 0.10, 2022: 0.15, 2023: 0.20, 2024: 0.25, 2025: 0.30}
        _seasons_cv = val_seasons  # [2021,2022,2023,2024,2025]
        _ens_scores = cv_results['ensemble']
        if len(_ens_scores) == len(_seasons_cv):
            _weighted_cv = sum(_ens_scores[i] * _cv_w.get(_seasons_cv[i], 0.2)
                               for i in range(len(_seasons_cv)))
            print(f"  [{gender_tag}] 加权CV Brier (2025×0.30…2021×0.10) = {_weighted_cv:.5f}")
            print(f"  [{gender_tag}] 等权CV Brier = {np.mean(_ens_scores):.5f}")
    else:
        best_alpha = 0.95   # 无OOF时使用经验值
        print(f"\n  [{gender_tag}] 无OOF记录，使用逆Brier权重，温度 T = {best_T:.3f} (默认)")

    # ---- P6-FIX1: Final full-data training with 2019 early stopping ----
    print(f"\n  [{gender_tag}] P6-FIX1: 最终全量模型训练 (2019验证集 early_stopping 100轮)...")
    df_final, quality_map_final = build_final_glm_quality_map(df, static_feat_cols)
    final_feat_cols = static_feat_cols + [c for c in GLM_NEW_COLS if c in df_final.columns]

    all_is_playoff = df_final.get('IsPlayoff', pd.Series(0, index=df_final.index)).values.astype(bool)
    all_weights = _make_sample_weights(df_final['Season'].values, all_is_playoff)

    # 切 2019 验证集（防过拟合）
    val_final = df_final[df_final['Season'] == 2019].copy()
    train_final = df_final[df_final['Season'] < 2019].copy()

    has_2019 = (len(val_final) > 0 and len(train_final) > 0)

    if not has_2019:
        # 2019 不存在时回退全量 + mean best iters
        print(f"  [{gender_tag}] ⚠️ 2019数据不存在，回退到全量训练")
        X_all_f = df_final[final_feat_cols].fillna(0)
        y_all_f = df_final['PointDiff'].values.astype(float)
        final_cal = SimpleCalibrator(n_bins=20, min_samples=20)
        final_cal.fit(y_all_f, df_final['Result'].values)
        num_rounds = max(200, int(np.mean(lgb_best_iters) * 1.1)) if lgb_best_iters else 1500
        lgb_final = lgb.train(lgb_params,
                              lgb.Dataset(X_all_f, label=y_all_f, weight=all_weights),
                              num_boost_round=num_rounds)
        xgb_params_fb = {k: v for k, v in xgb_cv_params.items()
                         if k not in ('early_stopping_rounds', 'n_estimators')}
        xgb_final = xgb.XGBRegressor(**xgb_params_fb, n_estimators=num_rounds)
        xgb_final.fit(X_all_f, y_all_f, sample_weight=all_weights, verbose=False)
    else:
        X_tr_f = train_final[final_feat_cols].fillna(0)
        y_tr_f = train_final['PointDiff'].values.astype(float)
        X_va_f = val_final[final_feat_cols].fillna(0)
        y_va_f = val_final['PointDiff'].values.astype(float)

        final_cal = SimpleCalibrator(n_bins=20, min_samples=20)
        final_cal.fit(y_tr_f, train_final['Result'].values)

        # LGB final with early stopping on 2019
        # 注意：final模型用更低学习率(0.02)，防止在2019验证集上过早触发early stopping
        # (CV阶段lr=0.04，best_iter~269，说明高lr导致模型跑太快，实际欠拟合)
        lgb_params_final = dict(lgb_params)
        lgb_params_final['learning_rate'] = 0.015  # v11: 0.02→0.015，期望best_iter 529→700-900
        print(f"  [{gender_tag}] LGB final lr=0.015 (更充分训练，期望iter↑)")
        tr_all = lgb.Dataset(X_tr_f, label=y_tr_f, weight=all_weights[train_final.index])
        va_all = lgb.Dataset(X_va_f, label=y_va_f, reference=tr_all)
        lgb_final = lgb.train(
            lgb_params_final, tr_all, num_boost_round=3000,
            valid_sets=[va_all],
            callbacks=[lgb.early_stopping(100, verbose=False)]
        )
        print(f"  [{gender_tag}] LGB final best_iteration = {lgb_final.best_iteration} (lr=0.015)")

        # XGB final with early stopping on 2019（修复参数重复）
        xgb_params_final = {k: v for k, v in xgb_cv_params.items()
                            if k not in ('early_stopping_rounds', 'n_estimators')}
        xgb_final = xgb.XGBRegressor(**xgb_params_final, n_estimators=3000, early_stopping_rounds=100)
        xgb_final.fit(
            X_tr_f, y_tr_f,
            sample_weight=all_weights[train_final.index],
            eval_set=[(X_va_f, y_va_f)],
            verbose=False
        )
        print(f"  [{gender_tag}] XGB final best_iteration = {xgb_final.best_iteration}")

    # CatBoost final (use 2019 val if available)
    cat_final = _build_catboost_model({})
    X_all_cat = df_final[final_feat_cols].fillna(0)
    y_all_cat = df_final['PointDiff'].values.astype(float)
    if HAS_CATBOOST and has_2019:
        X_tr_f_cat = train_final[final_feat_cols].fillna(0)
        y_tr_f_cat = train_final['PointDiff'].values.astype(float)
        X_va_f_cat = val_final[final_feat_cols].fillna(0)
        y_va_f_cat = val_final['PointDiff'].values.astype(float)
        cat_final.fit(X_tr_f_cat, y_tr_f_cat,
                      sample_weight=all_weights[train_final.index],
                      eval_set=(X_va_f_cat, y_va_f_cat), verbose=False)
    elif HAS_CATBOOST:
        cat_final.fit(X_all_cat, y_all_cat, sample_weight=all_weights, verbose=False)
    else:
        cat_final.fit(X_all_cat, y_all_cat, sample_weight=all_weights)

    tag = gender_tag.lower()
    joblib.dump(lgb_final, f'{WORKING}lgb_final_{tag}.pkl')
    joblib.dump(xgb_final, f'{WORKING}xgb_final_{tag}.pkl')
    joblib.dump(cat_final, f'{WORKING}cat_final_{tag}.pkl')
    joblib.dump(final_cal, f'{WORKING}calibrator_{tag}.pkl')
    print(f"  [{gender_tag}] ✅ 模型与校准器已保存到 {WORKING}")

    return {
        'gender': gender_tag,
        'cv_results': cv_results,
        'opt_weights': opt_weights,
        'inv_brier_weights': inv_brier_weights,
        'calibrator': final_cal,
        'lgb_final': lgb_final,
        'xgb_final': xgb_final,
        'cat_final': cat_final,
        'feature_cols': final_feat_cols,
        'static_feat_cols': static_feat_cols,
        'lgb_best_iters': lgb_best_iters,
        'quality_map_final': quality_map_final,
        'best_T': best_T,
        'best_alpha': best_alpha,   # P1(v9): OOF优化shrink alpha
    }


# -----------------------
# Build 2026 test features (P5-FIX7)
# -----------------------
def build_2026_test_features(df_full: pd.DataFrame, pipeline: Dict,
                             submission_template: pd.DataFrame,
                             gender_flag: int) -> Tuple[np.ndarray, pd.DataFrame]:
    feature_cols = pipeline['feature_cols']
    quality_map = pipeline.get('quality_map_final', pd.DataFrame())

    def _get_team_state(season: int):
        sub = df_full[(df_full['Season'] == season) & (df_full['is_mens'] == gender_flag)].copy()
        if sub.empty:
            return pd.DataFrame()
        return sub.sort_values('DayNum').groupby('TeamID').last().reset_index().set_index('TeamID')

    ts_2025 = _get_team_state(2025)
    ts_2024 = _get_team_state(2024)

    qm = quality_map.copy()
    glm_global_mean = float(qm['glm_quality'].mean()) if not qm.empty else 0.0
    qm25 = qm[qm['Season'] == 2025] if not qm.empty else pd.DataFrame()
    qm24 = qm[qm['Season'] == 2024] if not qm.empty else pd.DataFrame()
    glm25 = qm25.set_index('TeamID')['glm_quality'].to_dict() if not qm25.empty else {}
    glm24 = qm24.set_index('TeamID')['glm_quality'].to_dict() if not qm24.empty else {}

    def _get(tid: int, col: str, default: float = 0.0):
        for ts in [ts_2025, ts_2024]:
            if not ts.empty and tid in ts.index and col in ts.columns:
                v = ts.at[tid, col]
                if pd.notna(v):
                    return float(v)
        return default

    records = []
    missing_2025 = set()
    for gid in submission_template['ID']:
        _, t1s, t2s = gid.split('_')
        t1, t2 = int(t1s), int(t2s)
        if int(str(t1).startswith('1')) != gender_flag:
            continue
        if not ts_2025.empty:
            if t1 not in ts_2025.index:
                missing_2025.add(t1)
            if t2 not in ts_2025.index:
                missing_2025.add(t2)
        row = {'_game_id': gid}
        for feat in feature_cols:
            if feat == 'elo_diff':
                row['elo_diff'] = _get(t1, 'pre_game_elo', 1500.0) - _get(t2, 'pre_game_elo', 1500.0)
            elif feat == 'is_mens':
                row['is_mens'] = float(gender_flag)
            elif feat == 'glm_team_quality':
                row['glm_team_quality'] = glm25.get(t1, glm24.get(t1, glm_global_mean))
            elif feat == 'glm_opp_quality':
                row['glm_opp_quality'] = glm25.get(t2, glm24.get(t2, glm_global_mean))
            elif feat == 'glm_quality_diff':
                q1 = glm25.get(t1, glm24.get(t1, glm_global_mean))
                q2 = glm25.get(t2, glm24.get(t2, glm_global_mean))
                row['glm_quality_diff'] = q1 - q2
            elif feat.startswith('opp_'):
                row[feat] = _get(t2, feat[4:], 0.0)
            else:
                row[feat] = _get(t1, feat, 0.0)
        records.append(row)

    if missing_2025:
        print(f"  ⚠️ {len(missing_2025)} 支队伍2025数据缺失，已回退2024或均值 (P5-FIX7)")

    test_df = pd.DataFrame(records)
    for c in [f for f in feature_cols if f not in test_df.columns]:
        test_df[c] = 0.0
    test_df = test_df[feature_cols].fillna(0)
    game_ids = test_df['_game_id'].values if '_game_id' in test_df.columns else \
        np.array([r['_game_id'] for r in records])
    return game_ids, test_df


# -----------------------
# Symmetry & postprocess
# -----------------------
def enforce_symmetry(sub_df: pd.DataFrame) -> pd.DataFrame:
    """P6-FIX6: p(A beats B) + p(B beats A) = 1"""
    df = sub_df.copy()
    id_to_idx = {row['ID']: i for i, row in df.iterrows()}
    processed = set()
    for i, row in df.iterrows():
        gid = row['ID']
        if gid in processed:
            continue
        parts = gid.split('_')
        if len(parts) != 3:
            continue
        rev = f"{parts[0]}_{parts[2]}_{parts[1]}"
        if rev in id_to_idx:
            j = id_to_idx[rev]
            p_ab = float(df.at[i, 'Pred'])
            p_ba = float(df.at[j, 'Pred'])
            p_new = float(np.clip(0.5 * (p_ab + 1.0 - p_ba), 1e-6, 1 - 1e-6))
            df.at[i, 'Pred'] = p_new
            df.at[j, 'Pred'] = 1.0 - p_new
            processed.add(gid)
            processed.add(rev)
    return df


def temperature_scale_probs(probs: np.ndarray, T: float = 1.10) -> np.ndarray:
    eps = 1e-7
    p_clip = np.clip(probs, eps, 1 - eps)
    logits = np.log(p_clip / (1 - p_clip))
    return np.clip(1.0 / (1.0 + np.exp(-logits / T)), 1e-6, 1 - 1e-6)


def shrink_probs(p: np.ndarray, alpha: float = 0.95) -> np.ndarray:
    """
    ChatGPT/GM 概率收缩技巧：把预测概率向 0.5 收缩一点。
    公式: p_shrunk = 0.5 + (p - 0.5) * alpha
    alpha=0.95: 0.99→0.9455, 0.97→0.9215, 0.50→0.50 (不变)
    原理: Brier Score 对 overconfidence 的惩罚远大于 underconfidence。
         当真实爆冷时 (p=0.99, y=0), 损失=(0.99)^2=0.9801
         收缩后: (0.9455)^2=0.8940, 节省 0.086
    等价于在 temperature scaling 上叠加 T≈1/0.95≈1.053 的软化。
    """
    return np.clip(0.5 + (np.asarray(p) - 0.5) * alpha, 1e-6, 1 - 1e-6).astype(float)


def postprocess_submission(df_sub: pd.DataFrame,
                           clip_lo: float = 0.04, clip_hi: float = 0.96) -> pd.DataFrame:
    df = df_sub.copy()
    df['Pred'] = np.clip(df['Pred'].astype(float), clip_lo, clip_hi)
    return df


# -----------------------
# P6-FIX6: Generate submissions
# -----------------------
def generate_all_submissions(df_full: pd.DataFrame, pipeline_m: Dict, pipeline_w: Dict,
                             submission_template: pd.DataFrame):
    subs = {'lgb': [], 'xgb': [], 'cat': [], 'stack': []}

    for gender_tag, pipeline in [('M', pipeline_m), ('W', pipeline_w)]:
        tag = 1 if gender_tag == 'M' else 0
        game_ids, test_df = build_2026_test_features(df_full, pipeline, submission_template, tag)
        if len(game_ids) == 0:
            continue

        feat_cols = pipeline['feature_cols']
        X_test = test_df[feat_cols].fillna(0)

        final_cal = pipeline['calibrator']
        p_lgb = final_cal.predict(pipeline['lgb_final'].predict(X_test))
        p_xgb = final_cal.predict(pipeline['xgb_final'].predict(X_test))
        p_cat = final_cal.predict(pipeline['cat_final'].predict(X_test))

        w = pipeline['opt_weights']
        # v9: OOF动态T + OOF动态alpha（P1） + clip[0.025,0.975]
        gender_T     = pipeline.get('best_T',     SUBMISSION_TEMPERATURE)
        gender_alpha = pipeline.get('best_alpha', 0.95)
        shrink_note  = "（不收缩）" if gender_alpha >= 0.999 else f"OOF收缩 alpha={gender_alpha:.4f}"
        print(f"  [{gender_tag}] 使用温度 T = {gender_T:.3f}，{shrink_note}")
        p_stack_raw = w['xgb'] * p_xgb + w['lgb'] * p_lgb + w['cat'] * p_cat
        p_stack = shrink_probs(temperature_scale_probs(p_stack_raw, T=gender_T), alpha=gender_alpha)

        ids = list(game_ids)
        subs['lgb'].append(pd.DataFrame({'ID': ids, 'Pred': p_lgb}))
        subs['xgb'].append(pd.DataFrame({'ID': ids, 'Pred': p_xgb}))
        subs['cat'].append(pd.DataFrame({'ID': ids, 'Pred': p_cat}))
        subs['stack'].append(pd.DataFrame({'ID': ids, 'Pred': p_stack}))

    empty = pd.DataFrame(columns=['ID', 'Pred'])
    sub_lgb   = pd.concat(subs['lgb'],   ignore_index=True) if subs['lgb']   else empty.copy()
    sub_xgb   = pd.concat(subs['xgb'],   ignore_index=True) if subs['xgb']   else empty.copy()
    sub_cat   = pd.concat(subs['cat'],   ignore_index=True) if subs['cat']   else empty.copy()
    sub_stack = pd.concat(subs['stack'], ignore_index=True) if subs['stack'] else empty.copy()

    # P6-FIX6: shrink + clip[0.025,0.975] + enforce_symmetry (主提交)
    if not sub_stack.empty:
        sub_stack = postprocess_submission(sub_stack, 0.025, 0.975)
        sub_stack = enforce_symmetry(sub_stack)

    # BUG-3 FIX: 直接赋值替换 for 循环写法
    sub_lgb = enforce_symmetry(postprocess_submission(sub_lgb, 0.025, 0.975)) if not sub_lgb.empty else sub_lgb
    sub_xgb = enforce_symmetry(postprocess_submission(sub_xgb, 0.025, 0.975)) if not sub_xgb.empty else sub_xgb
    sub_cat = enforce_symmetry(postprocess_submission(sub_cat, 0.025, 0.975)) if not sub_cat.empty else sub_cat

    sub_lgb.to_csv(f'{WORKING}submission_lgb_v6.csv', index=False)
    sub_xgb.to_csv(f'{WORKING}submission_xgb_v6.csv', index=False)
    sub_cat.to_csv(f'{WORKING}submission_cat_v6.csv', index=False)
    sub_stack.to_csv(f'{WORKING}submission_stack_v6.csv', index=False)
    print(f"  ✅ 提交文件已保存 (主提交: submission_stack_v6.csv)")

    return {'lgb': sub_lgb, 'xgb': sub_xgb, 'cat': sub_cat, 'stack': sub_stack}


# -----------------------
# Main orchestration
# -----------------------
def main():
    # Load doubao features
    df_all = pd.read_parquet(FEATURE_PATH)
    print(f"Loaded doubao feature matrix: {df_all.shape}")

    # Ensure target exists
    if 'PointDiff' not in df_all.columns or 'Result' not in df_all.columns:
        games_list = []
        for prefix, fname in [('M', 'MRegularSeasonDetailedResults.csv'),
                               ('W', 'WRegularSeasonDetailedResults.csv'),
                               ('M', 'MNCAATourneyDetailedResults.csv'),   # 锦标赛（必须！）
                               ('W', 'WNCAATourneyDetailedResults.csv')]:
            fpath = os.path.join(BASE_PATH, fname)
            if not os.path.exists(fpath):
                continue
            r = pd.read_csv(fpath)
            for side in [
                r.rename(columns={'WTeamID': 'TeamID', 'LTeamID': 'OppTeamID',
                                   'WScore': 'Score', 'LScore': 'OppScore'}),
                r.rename(columns={'LTeamID': 'TeamID', 'WTeamID': 'OppTeamID',
                                   'LScore': 'Score', 'WScore': 'OppScore'}),
            ]:
                games_list.append(side)
        all_games = pd.concat(games_list, ignore_index=True)
        all_games['PointDiff'] = (all_games['Score'] - all_games['OppScore']).astype('float32')
        all_games['Result'] = (all_games['PointDiff'] > 0).astype('int8')
        df_all = df_all.merge(
            all_games[['Season', 'DayNum', 'TeamID', 'OppTeamID', 'PointDiff', 'Result']],
            on=['Season', 'DayNum', 'TeamID', 'OppTeamID'], how='left'
        )
        if df_all['PointDiff'].isna().sum() > 0:
            raise AssertionError("PointDiff merge failed and contains NaNs.")
        print("  ✅ 目标变量合并完成")

    # ---------------------------------------------------------------
    # IsPlayoff 验证（doubao_features.py 已正确写入，此处仅做校验）
    # ---------------------------------------------------------------
    if 'IsPlayoff' not in df_all.columns:
        df_all['IsPlayoff'] = np.int8(0)
        print("  ⚠️ IsPlayoff 列缺失（旧版parquet），已全部置0。请重新运行 doubao_features.py！")
    n_playoff = int(df_all['IsPlayoff'].sum())
    print(f"  ✅ IsPlayoff 列验证完成，共 {n_playoff} 场锦标赛样本（权重×3.5 已生效）")

    # Feature engineering (P5-FIX3)
    df_all, _ = add_feature_interactions(df_all)

    # P6-FIX4: 常规赛最后10场特征 (shift(1) 防泄露)
    if 'Score' in df_all.columns and 'OppScore' in df_all.columns:
        df_all = df_all.sort_values(['Season', 'TeamID', 'DayNum']).reset_index(drop=True)
        df_all['last10_score'] = (
            df_all.groupby(['Season', 'TeamID'])['Score']
            .transform(lambda x: x.rolling(10, min_periods=1).mean().shift(1))
        ).astype('float32')
        df_all['last10_opp_score'] = (
            df_all.groupby(['Season', 'TeamID'])['OppScore']
            .transform(lambda x: x.rolling(10, min_periods=1).mean().shift(1))
        ).astype('float32')
        df_all['last10_margin'] = (df_all['last10_score'] - df_all['last10_opp_score']).astype('float32')
        print("  ✅ P6-FIX4: last10_score / last10_opp_score / last10_margin 已添加")
    else:
        print("  ⚠️ P6-FIX4: Score/OppScore 不存在，跳过 last10 特征")

    # P6-FIX3: 种子历史胜率特征
    if 'seed_diff' in df_all.columns:
        df_all['seed_hist_win_rate'] = (
            df_all['seed_diff'].map(SEED_WIN_RATE_DICT).fillna(0.5).astype('float32')
        )
        print("  ✅ P6-FIX3: seed_hist_win_rate 已添加")
    else:
        print("  ⚠️ P6-FIX3: seed_diff 不存在，跳过 seed_hist_win_rate")

    # P6-FIX2: 历史锦标赛表现特征 (在性别拆分之前)
    tourney_feats = build_tourney_history_features(BASE_PATH)
    if not tourney_feats.empty:
        df_all = df_all.merge(tourney_feats, on=['Season', 'TeamID'], how='left')
        df_all['tourney_past_wins'] = df_all['tourney_past_wins'].fillna(0).astype('float32')
        df_all['tourney_past_final4'] = df_all['tourney_past_final4'].fillna(0).astype('float32')
        print("  ✅ P6-FIX2: tourney_past_wins / tourney_past_final4 已合并")

    # Conference strength
    conf_df = build_conference_strength(BASE_PATH)
    conf_df = conf_df.rename(columns={'conf_strength': 'team_conf_strength',
                                      'conf_rank': 'team_conf_rank'})
    # 合并 TeamID 的联赛强度
    df_all = df_all.merge(conf_df, on=['Season', 'TeamID'], how='left')
    for col in ['team_conf_strength', 'team_conf_rank']:
        if col in df_all.columns:
            df_all[col] = df_all[col].fillna(0.5).astype('float32')

    # BUG-1 FIX: 合并 OppTeamID 的联赛强度（原代码用 df.get() 回退 0.5，对手信息完全丢失）
    opp_conf_df = conf_df.rename(columns={
        'TeamID': 'OppTeamID',
        'team_conf_strength': 'opp_team_conf_strength',
        'team_conf_rank': 'opp_team_conf_rank'
    })
    df_all = df_all.merge(
        opp_conf_df[['Season', 'OppTeamID', 'opp_team_conf_strength']],
        on=['Season', 'OppTeamID'], how='left'
    )
    df_all['opp_team_conf_strength'] = df_all['opp_team_conf_strength'].fillna(0.5).astype('float32')

    # P6-FIX5: conf_strength_diff (现在正确使用对手联赛强度)
    if 'team_conf_strength' in df_all.columns:
        df_all['conf_strength_diff'] = (
            df_all['team_conf_strength'] - df_all['opp_team_conf_strength']
        ).fillna(0).astype('float32')
        print("  ✅ P6-FIX5: conf_strength_diff 已修复（对手联赛强度正确合并）")

    # Split genders
    df_m = df_all[df_all['is_mens'] == 1].copy().reset_index(drop=True)
    df_w = df_all[df_all['is_mens'] == 0].copy().reset_index(drop=True)
    print(f"男子 rows: {len(df_m)}  女子 rows: {len(df_w)}")

    # P6-FIX5: dynamic top40 feature selection → save _v6.csv
    feat_m = select_top_features_dynamic(df_m, 'M', top_n=50)
    feat_w = select_top_features_dynamic(df_w, 'W', top_n=50)

    # Train both genders (P6-FIX1 inside)
    pipeline_m = train_gender_pipeline(df_m, feat_m, 'M',
                                       best_lgb_params={}, best_xgb_params={}, n_optuna_trials=0)
    pipeline_w = train_gender_pipeline(df_w, feat_w, 'W',
                                       best_lgb_params={}, best_xgb_params={}, n_optuna_trials=0)

    # Load submission template — BUG-5 FIX: 尝试多个可能的文件名
    sample_path = None
    for _sname in ['SampleSubmissionStage2.csv', 'SampleSubmissionStage1.csv',
                   'SampleSubmission.csv', 'sample_submission.csv']:
        _p = os.path.join(BASE_PATH, _sname)
        if os.path.exists(_p):
            sample_path = _p
            print(f"  ✅ 找到提交模板: {_sname}")
            break
    if sample_path is None:
        raise FileNotFoundError(
            f"❌ 未找到提交模板文件，已尝试: SampleSubmissionStage1.csv / SampleSubmission.csv / Stage2"
        )
    submission_template = pd.read_csv(sample_path)

    # P6-FIX6: OOF 动态 T + enforce_symmetry + clip[0.04,0.96]
    generate_all_submissions(df_all, pipeline_m, pipeline_w, submission_template)

    print("\n✅ ncaa_2026_gold_medal_v6.py 完成。主提交: submission_stack_v6.csv")
    print("   预期 Brier 目标: 0.186 ~ 0.189")


if __name__ == '__main__':
    main()

✅ CatBoost 可用
Loaded doubao feature matrix: (406000, 51)
  ✅ 目标变量合并完成
  ✅ IsPlayoff 列验证完成，共 4820 场锦标赛样本（权重×3.5 已生效）
  ✅ 特征工程 2.0 新增 16 个特征 (P5-FIX3 applied)
  ✅ P6-FIX4: last10_score / last10_opp_score / last10_margin 已添加
  ✅ P6-FIX3: seed_hist_win_rate 已添加
  ✅ P6-FIX2 (稳健版): 历史锦标赛特征构建完成，4369 个队季度记录
  ✅ P6-FIX2: tourney_past_wins / tourney_past_final4 已合并
  ✅ P6-FIX5: conf_strength_diff 已修复（对手联赛强度正确合并）
男子 rows: 240662  女子 rows: 165338

  [M] 动态筛选 Top50 特征完成，前5: ['elo_diff', 'conf_strength_diff', 'elo_diff_sq', 'opp_pre_game_elo', 'last10_margin']
  [M] 特征重要性已保存 → feature_importance_M_v6.csv

  [W] 动态筛选 Top50 特征完成，前5: ['elo_diff', 'conf_strength_diff', 'elo_diff_sq', 'last10_margin', 'opp_pre_game_elo']
  [W] 特征重要性已保存 → feature_importance_W_v6.csv

男子 [M] training v6
  [M] Season 2021: XGB=0.31789 LGB=0.31852 CAT=0.31801 Ens=0.30828
  [M] Season 2022: XGB=0.29110 LGB=0.29137 CAT=0.29174 Ens=0.28356
  [M] Season 2023: XGB=0.30630 LGB=0.30692 CAT=0.30559 Ens=0.29829
  [M] Season 2024: XGB